**Table of contents**<a id='toc0_'></a>    
- [04 — Build a Vanilla RNN Seq2Seq Model](#toc1_)    
  - [Load the processed parallel corpus](#toc1_1_)    
  - [Inspect a real training batch and create source embeddings](#toc1_2_)    
  - [Define one vanilla RNN step](#toc1_3_)    
  - [Read each source sentence to its final valid hidden state](#toc1_4_)    
  - [Check that extra source padding changes nothing](#toc1_5_)    
  - [Define one decoder step](#toc1_6_)    
  - [Unroll the decoder with Teacher Forcing](#toc1_7_)    
  - [Compute cross-entropy over real target tokens](#toc1_8_)    
  - [Combine encoder and decoder and verify backpropagation](#toc1_9_)    
  - [Perform one real-batch parameter update](#toc1_10_)    
  - [Move the model and a real batch to the training device](#toc1_11_)    
  - [Define train and development passes](#toc1_12_)    
  - [Run one full training epoch when ready](#toc1_13_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[04 Build a Vanilla RNN Seq2Seq Model](#toc0_)

## <a id='toc1_1_'></a>[Load the processed parallel corpus](#toc0_)

Load the project-local English–Vietnamese data. The saved ID pairs and vocabularies come from preprocessing; all model components below are defined in this notebook.

In [1]:
import sys
from pathlib import Path

import torch
from torch import nn

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_corpus, make_loaders
corpus = load_corpus(PROJECT_ROOT / "data")

## <a id='toc1_2_'></a>[Inspect a real training batch and create source embeddings](#toc0_)

The training loader covers every saved training pair. We inspect one shuffled batch to check tensor shapes. An integer token ID becomes an embedding vector $e^{\langle t\rangle}$ of width 128; the RNN input is $x^{\langle t\rangle}=e^{\langle t\rangle}$. Padding ID 0 has a zero embedding.

In [2]:
train_loader, dev_loader, test_loader = make_loaders(corpus, batch_size=64)
src_padded, src_lengths, tgt_input, tgt_output = next(iter(train_loader))
embed_dim = 128
encoder_embedding = nn.Embedding(
    num_embeddings=len(corpus.en_vocab),
    embedding_dim=embed_dim,
    padding_idx=0,
)

src_embeddings = encoder_embedding(src_padded)

print("Training pairs:", len(corpus.train_pairs))
print("Training batches:", len(train_loader))
print("Source IDs:", src_padded.shape)
print("Source embeddings:", src_embeddings.shape)
print("Source lengths:", src_lengths.shape)

# The DataLoader visits every training pair, including the smaller final batch.

Training pairs: 125798
Training batches: 1966
Source IDs: torch.Size([64, 51])
Source embeddings: torch.Size([64, 51, 128])
Source lengths: torch.Size([64])


## <a id='toc1_3_'></a>[Define one vanilla RNN step](#toc0_)

With column-vector notation, $z_h^{\langle t\rangle}=W_{hx}x^{\langle t\rangle}+W_{hh}h^{\langle t-1\rangle}+b_h$ and $h^{\langle t\rangle}=\tanh(z_h^{\langle t\rangle})$. PyTorch batches contain row vectors, so code multiplies by the transposed weight matrices. The two-step check uses token embeddings from the real batch.

In [3]:
class VanillaRNNCell(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()

        # Column-vector notation: W_hx [H, D], W_hh [H, H].
        self.W_hx = nn.Parameter(torch.empty(hidden_dim, embed_dim))
        self.W_hh = nn.Parameter(torch.empty(hidden_dim, hidden_dim))
        self.b_h = nn.Parameter(torch.zeros(hidden_dim))

        nn.init.xavier_uniform_(self.W_hx)
        nn.init.orthogonal_(self.W_hh)

    def forward(self, x_t, h_prev):
        z_h = x_t @ self.W_hx.T + h_prev @ self.W_hh.T + self.b_h
        h_t = torch.tanh(z_h)
        return h_t

hidden_dim = 256
encoder_cell = VanillaRNNCell(embed_dim, hidden_dim)
batch_size = src_embeddings.shape[0]

# Initial hidden state for all sentences in this real batch.
h_0 = src_embeddings.new_zeros(batch_size, hidden_dim)

h_1 = encoder_cell(src_embeddings[:, 0, :], h_0)
h_2 = encoder_cell(src_embeddings[:, 1, :], h_1)

print("Input at timestep 0:", src_embeddings[:, 0, :].shape)
print("Initial hidden state:", h_0.shape)
print("Hidden after timestep 0:", h_1.shape)
print("Hidden after timestep 1:", h_2.shape)

Input at timestep 0: torch.Size([64, 128])
Initial hidden state: torch.Size([64, 256])
Hidden after timestep 0: torch.Size([64, 256])
Hidden after timestep 1: torch.Size([64, 256])


## <a id='toc1_4_'></a>[Read each source sentence to its final valid hidden state](#toc0_)

The English encoder shares its RNN weights across source positions. `src_lengths` includes `<bos>` and `<eos>`. For each sentence, padding positions retain the previous hidden state, so the returned `[B, H]` tensor contains the state after its final real token.

In [4]:
class Encoder(nn.Module):
    def __init__(self, embedding, cell):
        super().__init__()
        self.embedding = embedding
        self.cell = cell

    def forward(self, src_padded, src_lengths):
        embeddings = self.embedding(src_padded)
        batch_size, num_steps, _ = embeddings.shape
        hidden_dim = self.cell.W_hh.shape[0]

        h_t = embeddings.new_zeros(batch_size, hidden_dim)

        for t in range(num_steps):
            candidate = self.cell(embeddings[:, t, :], h_t)
            # Keep the previous state after a sentence reaches padding.
            is_real_token = (t < src_lengths).unsqueeze(1)
            h_t = torch.where(is_real_token, candidate, h_t)

        return h_t


encoder = Encoder(encoder_embedding, encoder_cell)
final_hidden = encoder(src_padded, src_lengths)
print("Source lengths:", src_lengths[:5].tolist())
print("Final encoder states:", final_hidden.shape)


Source lengths: [9, 17, 9, 21, 46]
Final encoder states: torch.Size([64, 256])


## <a id='toc1_5_'></a>[Check that extra source padding changes nothing](#toc0_)

Append five `<pad>` IDs to the real source batch while keeping the original lengths. The final encoder states should match exactly.

In [5]:
src_with_extra_padding = torch.nn.functional.pad(
    src_padded,
    pad=(0, 5),
    value=0,
)

with torch.no_grad():
    hidden_original = encoder(src_padded, src_lengths)
    hidden_extra_padding = encoder(src_with_extra_padding, src_lengths)

print("Original source shape:", src_padded.shape)
print("Extended source shape:", src_with_extra_padding.shape)
print(
    "All final states unchanged:",
    torch.allclose(hidden_original, hidden_extra_padding),
)
print(
    "Largest difference:",
    (hidden_original - hidden_extra_padding).abs().max().item(),
)

Original source shape: torch.Size([64, 51])
Extended source shape: torch.Size([64, 56])
All final states unchanged: True
Largest difference: 0.0


## <a id='toc1_6_'></a>[Define one decoder step](#toc0_)

The Vietnamese decoder has its own embedding and RNN weights. Its initial state is the encoder final state. A decoder step consumes one target token ID and returns a new hidden state plus unnormalized output logits $z_y^{\langle t\rangle}=W_{yh}h_{\mathrm{dec}}^{\langle t\rangle}+b_y$.

In [6]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0,
        )
        self.cell = VanillaRNNCell(embed_dim, hidden_dim)

        self.W_yh = nn.Parameter(torch.empty(vocab_size, hidden_dim))
        self.b_y = nn.Parameter(torch.zeros(vocab_size))
        nn.init.xavier_uniform_(self.W_yh)

    def step(self, token_ids, h_prev):
        x_t = self.embedding(token_ids)
        h_t = self.cell(x_t, h_prev)
        logits_t = h_t @ self.W_yh.T + self.b_y
        return logits_t, h_t


decoder = Decoder(
    vocab_size=len(corpus.vi_vocab),
    embed_dim=embed_dim,
    hidden_dim=hidden_dim,
)

first_decoder_tokens = tgt_input[:, 0]
first_logits, first_decoder_hidden = decoder.step(
    first_decoder_tokens,
    final_hidden,
)

print("Decoder input IDs:", first_decoder_tokens.shape)
print("First input is <bos>:", torch.all(first_decoder_tokens == 1).item())
print("Decoder hidden:", first_decoder_hidden.shape)
print("Logits:", first_logits.shape)
print("Reference next-token IDs:", tgt_output[:, 0].shape)

Decoder input IDs: torch.Size([64])
First input is <bos>: True
Decoder hidden: torch.Size([64, 256])
Logits: torch.Size([64, 16254])
Reference next-token IDs: torch.Size([64])


## <a id='toc1_7_'></a>[Unroll the decoder with Teacher Forcing](#toc0_)

At each target position, feed the reference token from `tgt_input`. This is 100% Teacher Forcing. The output shape is `[B, T_tgt, V_vi]`; `tgt_output` contains the corresponding next-token IDs. This inspection uses `no_grad()` to avoid retaining an unnecessary graph.

In [7]:
# This cell checks the forward pass only; training will enable gradients.
with torch.no_grad():
    h_t = final_hidden
    logits_by_step = []

    for t in range(tgt_input.shape[1]):
        token_ids = tgt_input[:, t]
        logits_t, h_t = decoder.step(token_ids, h_t)
        logits_by_step.append(logits_t)

    logits = torch.stack(logits_by_step, dim=1)

print("Decoder input:", tgt_input.shape)
print("Decoder target:", tgt_output.shape)
print("All logits:", logits.shape)
print("First input token:", tgt_input[0, 0].item())
print("First target token:", tgt_output[0, 0].item())

assert logits.shape[:2] == tgt_output.shape

Decoder input: torch.Size([64, 51])
Decoder target: torch.Size([64, 51])
All logits: torch.Size([64, 51, 16254])
First input token: 1
First target token: 6


## <a id='toc1_8_'></a>[Compute cross-entropy over real target tokens](#toc0_)

`CrossEntropyLoss` receives raw logits and target IDs. `ignore_index=0` excludes `<pad>` targets; dividing the summed loss by the non-padding token count gives token-average loss. This cell only measures loss and does not train.

In [8]:
criterion = nn.CrossEntropyLoss(
    ignore_index=0,
    reduction="sum",
)

loss_sum = criterion(
    logits.reshape(-1, logits.shape[-1]),
    tgt_output.reshape(-1),
)

valid_tokens = (tgt_output != 0).sum()
loss_per_token = loss_sum / valid_tokens

print("Target positions:", tgt_output.numel())
print("Real target tokens:", valid_tokens.item())
print("Loss per real token:", loss_per_token.item())

Target positions: 3264
Real target tokens: 1351
Loss per real token: 9.705543518066406


## <a id='toc1_9_'></a>[Combine encoder and decoder and verify backpropagation](#toc0_)

The `Seq2Seq` module reruns the complete forward pass with gradients enabled. Backpropagation through time should create gradients for decoder output weights, encoder RNN weights, and source embeddings.

In [9]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src_padded, src_lengths, tgt_input):
        h_t = self.encoder(src_padded, src_lengths)
        logits_by_step = []

        for t in range(tgt_input.shape[1]):
            logits_t, h_t = self.decoder.step(tgt_input[:, t], h_t)
            logits_by_step.append(logits_t)

        return torch.stack(logits_by_step, dim=1)


model = Seq2Seq(encoder, decoder)

# Run again with gradients enabled.
model.zero_grad(set_to_none=True)
train_logits = model(src_padded, src_lengths, tgt_input)

loss_sum = criterion(
    train_logits.reshape(-1, train_logits.shape[-1]),
    tgt_output.reshape(-1),
)
loss = loss_sum / (tgt_output != 0).sum()
loss.backward()

print("Logits:", train_logits.shape)
print("Loss:", loss.item())
print("Encoder embedding has gradient:",
      model.encoder.embedding.weight.grad is not None)
print("Encoder W_hx has gradient:",
      model.encoder.cell.W_hx.grad is not None)
print("Decoder W_yh has gradient:",
      model.decoder.W_yh.grad is not None)

Logits: torch.Size([64, 51, 16254])
Loss: 9.705543518066406
Encoder embedding has gradient: True
Encoder W_hx has gradient: True
Decoder W_yh has gradient: True


## <a id='toc1_10_'></a>[Perform one real-batch parameter update](#toc0_)

Adam updates all registered parameters after backward propagation. Gradient clipping limits the global gradient norm to 1.0. Comparing the same batch before and after one update checks that optimization is connected. This small check runs on the current device before the later CUDA move.

In [10]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
optimizer.zero_grad(set_to_none=True)

logits_before = model(src_padded, src_lengths, tgt_input)
loss_before = criterion(
    logits_before.reshape(-1, logits_before.shape[-1]),
    tgt_output.reshape(-1),
) / (tgt_output != 0).sum()

loss_before.backward()
gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

with torch.no_grad():
    logits_after = model(src_padded, src_lengths, tgt_input)
    loss_after = criterion(
        logits_after.reshape(-1, logits_after.shape[-1]),
        tgt_output.reshape(-1),
    ) / (tgt_output != 0).sum()

print("Loss before update:", loss_before.item())
print("Gradient norm before clipping:", gradient_norm.item())
print("Loss after update:", loss_after.item())

Loss before update: 9.705543518066406
Gradient norm before clipping: 0.45220547914505005
Loss after update: 9.5847806930542


## <a id='toc1_11_'></a>[Move the model and a real batch to the training device](#toc0_)

Use CUDA when available. Model parameters and all batch tensors must reside on the same device. The optimizer from the CPU check is replaced after this move, because its existing Adam state belongs to the earlier device.

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

batch = next(iter(train_loader))
src_batch, lengths_batch, tgt_in_batch, tgt_out_batch = (
    tensor.to(device) for tensor in batch
)

with torch.no_grad():
    batch_logits = model(src_batch, lengths_batch, tgt_in_batch)

print("Device:", device)
print("Source device:", src_batch.device)
print("Model device:", next(model.parameters()).device)
print("Logits shape:", batch_logits.shape)
del batch_logits

Device: cuda
Source device: cuda:0
Model device: cuda:0
Logits shape: torch.Size([64, 49, 16254])


## <a id='toc1_12_'></a>[Define train and development passes](#toc0_)

`run_epoch` visits every batch in its loader. Training enables autograd and updates weights; development disables autograd and only measures loss. Each reported loss is the sum of cross-entropies divided by the number of non-padding target tokens.

In [12]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(model, loader, criterion, device, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)

    total_loss = 0.0
    total_tokens = 0

    for batch_index, batch in enumerate(loader, start=1):
        src, lengths, tgt_in, tgt_out = (
            tensor.to(device) for tensor in batch
        )

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(src, lengths, tgt_in)
            loss_sum = criterion(
                logits.reshape(-1, logits.shape[-1]),
                tgt_out.reshape(-1),
            )
            valid_tokens = (tgt_out != 0).sum()

            if is_training:
                (loss_sum / valid_tokens).backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        total_loss += loss_sum.detach().item()
        total_tokens += valid_tokens.item()

        if is_training and batch_index % 100 == 0:
            print(
                f"Batch {batch_index}/{len(loader)} | "
                f"train loss: {total_loss / total_tokens:.4f}",
                flush=True,
            )

    return total_loss / total_tokens

KeyboardInterrupt: 

## <a id="toc1_13_"></a>[Train with checkpoint selection and early stopping](#toc0_)

Run this cell only when the training device is ready. Each epoch processes every training batch, then measures token-average loss on the development split. The checkpoint stores the model with the lowest development loss. Stop after three epochs without improvement or after ten epochs. The test split is not used here.

In [ ]:
checkpoint_path = PROJECT_ROOT / "checkpoints" / "best_notebook.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

max_epochs = 10
patience = 3
best_dev_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(1, max_epochs + 1):
    train_loss = run_epoch(
        model, train_loader, criterion, device, optimizer=optimizer
    )
    dev_loss = run_epoch(model, dev_loader, criterion, device)

    print(
        f"Epoch {epoch} | train loss: {train_loss:.4f} "
        f"| dev loss: {dev_loss:.4f}",
        flush=True,
    )

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        epochs_without_improvement = 0
        torch.save(
            {
                "epoch": epoch,
                "dev_loss": dev_loss,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "embed_dim": embed_dim,
                "hidden_dim": hidden_dim,
                "src_vocab_size": len(corpus.en_vocab),
                "tgt_vocab_size": len(corpus.vi_vocab),
            },
            checkpoint_path,
        )
        print(f"Saved best checkpoint: {checkpoint_path}", flush=True)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print("Early stopping: dev loss did not improve.", flush=True)
            break
